<a href="https://colab.research.google.com/github/daffiss/machine-learning-practices/blob/main/materials/practices/p01/p01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning — Practical P01: Experimental Design and Basic Leakage

**Course:** Machine Learning · National University of Kyiv-Mohyla Academy (NaUKMA)  
**Instructor:** Dmytro Kuzmenko · kuzmenko@ukma.edu.ua  
**Module:** 1. What Does It Mean to Learn? · **Week 1**  
**Workload:** one practical class ≈ 1.5–2 hours of active work, including discussion  
**Status:** NOT graded — practicals are skill-development and exam-preparation sessions.  
**Prerequisites:** basics of Python, numpy, and scikit-learn; the idea of train/test split from the lecture

### How to work with this notebook

Every consequential task follows the loop **Experiment → Decision → Evidence → Interpretation → Counterfactual**.  
Scaffolding cells are already written and executed: read them, run them mentally, and use the printed numbers and plots as evidence.  
Empty cells are yours — they contain no model answers. The discussion questions are meant to be talked through in class.


### Learning objectives
- Design a minimal valid experiment: baseline, split, and a single performance number.
- Diagnose overfitting from the train–test gap.
- Recognize three classic leakage patterns and know how to fix each.

## Warm-up (diagnostic)

Answer these three questions before looking at the guided exercise. There are no wrong answers at this
stage — the point is to surface your current intuitions.


**Warm-up 1.** A model for a medical screening task reports **accuracy 98.4%** on the test set.
The prevalence of the positive class in the data is **1.2%** (98.8% of all cases are negative).
Which statement is the most defensible?

- (a) The model is excellent: 98.4% is well above 95%.
- (b) A trivial baseline that always predicts the majority class would score ≈ 98.8%, so accuracy alone tells us almost nothing about this model's value.
- (c) Accuracy is the right metric here because the classes are almost balanced.
- (d) The model must be broken because accuracy cannot exceed 98.8%.


**Your answer:** b)

**Warm-up 2.** You have one row per hospital visit for 5,000 patients and split the rows randomly 80/20.
You then discover that some patients have visits in *both* the train and the test part. Why is that a problem?

- (a) It is not a problem: random splits are always valid.
- (b) The test part is no longer independent of the training part, so the test accuracy will typically be optimistically biased.
- (c) It only matters for deep learning models.
- (d) It inflates the training time.


**Your answer:** b)

**Warm-up 3.** You train a model and get **train accuracy 0.99, test accuracy 0.55**.
In one sentence: what does this gap tell you, and which of the two numbers should you report to a stakeholder?


**Your answer:** The model is severely overfitting (it memorized the training data instead of generalizing), so you should report the test accuracy of 0.55 (along with an explanation that the model does not yet perform well enough for deployment)

## Guided Exercise — baseline, gap, leakage

We will run one small synthetic experiment and then hunt for leaks in three short snippets.
Everything below runs in seconds — the point is the reasoning, not the computation.


### Task 1: Baseline vs model on synthetic data

**Experiment.** Generate 2,000 points from two overlapping Gaussians (a binary classification problem),
split them 80/20, and compare two predictors: a *majority-class baseline* (always predicts the most
frequent class) and a logistic regression model.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

np.random.seed(42)
%matplotlib inline

# Two overlapping Gaussian blobs -> inherently noisy, no perfect classifier exists
rng = np.random.RandomState(42)
n = 2000
X = np.vstack([rng.normal(loc=[1.0, 1.0], scale=1.0, size=(n // 2, 2)),
               rng.normal(loc=[2.2, 2.2], scale=1.0, size=(n // 2, 2))])
y = np.array([0] * (n // 2) + [1] * (n // 2))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline: always predict the majority class
majority_class = int(y_train.mean() >= 0.5)
base_acc = accuracy_score(y_test, np.full(len(y_test), majority_class))

# Model: logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test))

print(f"Train set: {len(X_train)} rows, test set: {len(X_test)} rows")
print(f"Share of class 1 in train: {y_train.mean():.3f}")
print(f"Majority-class baseline test accuracy: {base_acc:.3f}")
print(f"Logistic regression test accuracy:      {lr_acc:.3f}")


Train set: 1600 rows, test set: 400 rows
Share of class 1 in train: 0.499
Majority-class baseline test accuracy: 0.497
Logistic regression test accuracy:      0.767


**Evidence.** The baseline gets accuracy ≈ 0.50 on this balanced problem — chance level. The model
should beat it clearly. On real imbalanced problems the baseline can look much stronger.

**Interpretation.** Why do we *always* want the baseline number on the table, even when the model's
accuracy looks high? What could "accuracy 0.80 vs baseline 0.79" tell you?

**Decision.** Imagine the same experiment with a 5% positive class and model accuracy 0.94.
What is the minimum extra information you need before calling the model useful?


**Your answer:** якщо у вас 95% класу A і 5% класу B, baseline "завжди A" вже дає accuracy = 0.95. Якщо ваша "розумна" модель показує accuracy = 0.94 — вона насправді гірша за тупе вгадування, попри враження "94% — це ж дуже добре!". Без baseline на таблиці це легко не помітити і презентувати провальну модель як успішну.

### Task 2: The train–test gap as a diagnostic

**Experiment.** Fit k-nearest-neighbors with k = 1 (very high capacity) and k = 20 (smooth decision
boundary) on the same split, and compare train vs test accuracy.


In [ ]:
for k in [1, 20]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    tr = accuracy_score(y_train, knn.predict(X_train))
    te = accuracy_score(y_test, knn.predict(X_test))
    print(f"k={k:>2}: train {tr:.3f} | test {te:.3f} | gap {tr - te:+.3f}")


k= 1: train 1.000 | test 0.665 | gap +0.335
k=20: train 0.796 | test 0.755 | gap +0.041


**Evidence.** k = 1 memorizes the training points (train accuracy ≈ 1.0) and pays for it on the test set.

**Interpretation.** Which of the two models would you deploy, and what does the *gap* — not the train
number — tell you about k = 1?

**Counterfactual.** Suppose we multiplied the training set by 10 (same distribution). Predict what
happens to the train accuracy, the test accuracy, and the gap for k = 1, and justify in one sentence.


***Що таке k-nearest-neighbors (KNN)***

KNN — це "лінива" модель, яка не будує явну формулу, а просто запам'ятовує всі тренувальні точки. Щоб передбачити клас нової точки, вона:

Знаходить k найближчих сусідів серед тренувальних точок.
Дивиться, який клас переважає серед них.
Повертає цей клас як передбачення.

Параметр k — це "ручка складності" моделі:

Малий k (наприклад, k=1) → модель дуже чутлива до кожної окремої точки, межа рішення дуже "звивиста", підлаштовується під кожен шум.
Великий k → модель усереднює по більшій кількості сусідів, межа стає гладкою, стійкою до шуму.

*Gap — це вимірювач перенавчання (overfitting), а не якості моделі.*

Великий gap (як +0.335 при k=1) сигналізує: "модель вивчила шум і специфіку конкретних тренувальних точок, а не загальну закономірність. Довіряти train-числу не можна — воно завищене й нерепрезентативне для реального застосування."
Маленький gap (як +0.041 при k=20) сигналізує: "модель узгоджена — те, що вона вміє на train, приблизно те саме вміє і на нових даних. Її можна прогнозовано використовувати."

**Your answer:**  Suppose we multiplied the training set by 10 (same distribution, 16000). Predict what happens to the train accuracy, the test accuracy, and the gap for k = 1, and justify in one sentence.

k=1

I suppose train accuracy would be still 1.000, while test acc may get higher as there is more data and higher chance that a model can guees the answer from 1 neighbour. the gap i guess will be smaller.

### Task 3: Spotting leakage

Leakage means that information from the test (or from the future, or from the target) reaches the model
during training. Each snippet below contains one classic leak. Run it, look at the suspiciously good
number, and explain where the leak is.


**Snippet 3a — a feature that already contains the answer.**

The last column of `X_leaky` was built *from the labels* after they were created.


In [ ]:
X_leaky = np.hstack([X, y.reshape(-1, 1)])  # extra column == the label
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
lr2 = LogisticRegression(max_iter=1000)
lr2.fit(Xl_tr, yl_tr)
print(f"Test accuracy with the 'extra' feature: {accuracy_score(yl_te, lr2.predict(Xl_te)):.3f}")


Test accuracy with the 'extra' feature: 1.000


**Що таке leakage в принципі**

Leakage (витік даних) — це ситуація, коли модель під час навчання отримує доступ до інформації, якої в реальному, "бойовому" застосуванні у неї просто не буде. Це може бути:

інформація з тестового набору, що просочилась у тренувальний (наприклад, через неправильний препроцесинг до split'у);
інформація "з майбутнього" (наприклад, дані, які стають відомі тільки після події, яку ви намагаєтесь передбачити);
інформація, похідна від самого таргету (мітки) — саме цей випадок у Snippet 3a.

**Interpretation.** Why is this test accuracy meaningless, even though the model was fitted only on
`Xl_tr`? How would you spot this leak in a real dataset with 200 columns?


**Your answer:** Test accuracy безглузда, бо коректність спліту (модель дійсно бачила під час навчання тільки Xl_tr, а не Xl_te) захищає лише від leakage на рівні рядків — тобто від ситуації, коли тестові спостереження випадково потрапляють у train. Тут такого немає, спліт чесний.

Але leak стався раніше й на іншому рівні: "extra"-колонка була сконструйована з y ще до спліту, для всіх 2000 точок одразу. Тому і Xl_tr, і Xl_te містять цю саму "заражену" колонку, яка в кожному окремому рядку прямо кодує мітку цього ж рядка. Модель, навчена на Xl_tr, засвоює правило "extra ≈ 1 → клас 1" — і це правило автоматично спрацьовує ідеально й на Xl_te, бо там колонка побудована тим самим механізмом з тих самих правдивих міток. Модель не навчилась узагальнювати залежність y від незалежних ознак — вона навчилась розшифровувати вбудовану підказку, яка присутня однаково і в train, і в test. Тому accuracy = 1.000 не має жодного відношення до реальної здатності моделі передбачати y на нових даних, де такої підказкової колонки просто не буде.

How would you spot this leak in a real dataset with 200 columns?
Перевірити кореляцію/зв'язок кожної з 200 ознак з таргетом окремо, до навчання моделі (кореляція для числових, mutual information/χ² для категоріальних). Ознака з кореляцією, близькою до ±1, — головний підозрюваний.
Подивитись на feature importance / коефіцієнти навченої моделі — якщо 1-2 ознаки з 200 мають на порядки більшу вагу за решту, це сигнал.
Видалити підозрілу ознаку і перенавчити модель — якщо accuracy падає з ~1.000 до реалістичного рівня, причину знайдено.
Перевірити часове походження кожної підозрілої ознаки: чи могла ця величина бути реально відома на момент, коли треба робити передбачення, чи вона з'являється в системі лише після настання таргет-події (наприклад, "дата закриття рахунку" для задачі передбачення відтоку клієнтів).

**Snippet 3b — preprocessing fitted on the whole dataset.**

Here the test distribution is shifted (a new measurement device). One pipeline fits `StandardScaler` on
**train + test together** and only then splits; the other fits the scaler on the train part only.
The printed comparison includes the scaler's learned means and the fitted model coefficients, not just
the accuracy.


In [ ]:
n_tr, n_te = 1500, 500
rng = np.random.RandomState(7)
Xbase = rng.normal(loc=1.0, scale=1.0, size=(n_tr + n_te, 2))
ybase = (Xbase[:, 0] > 1.0).astype(int)
Xtr, Xte, ytr, yte = train_test_split(Xbase, ybase, test_size=n_te / (n_tr + n_te), random_state=42)
Xte = Xte + 2.0          # "new device": every test measurement is shifted by +2

Xfull, yfull = np.vstack([Xtr, Xte]), np.hstack([ytr, yte])

# LEAKY: scaler fitted on train + test, then the model is trained on the train rows
scaler_leaky = StandardScaler().fit(Xfull)
lrL = LogisticRegression(max_iter=1000).fit(scaler_leaky.transform(Xtr), ytr)
acc_leaky = accuracy_score(yte, lrL.predict(scaler_leaky.transform(Xte)))

# CORRECT: scaler fitted on train only
scaler = StandardScaler().fit(Xtr)
lrC = LogisticRegression(max_iter=1000).fit(scaler.transform(Xtr), ytr)
acc_correct = accuracy_score(yte, lrC.predict(scaler.transform(Xte)))

print(f"scaler mean of feature 0: pooled (leak) {scaler_leaky.mean_[0]:.3f} | train only (ok) {scaler.mean_[0]:.3f}")
print(f"model coefficients:        leaky  {np.round(lrL.coef_[0], 3)} | correct {np.round(lrC.coef_[0], 3)}")
print(f"test accuracy:             leaky  {acc_leaky:.3f} | correct {acc_correct:.3f}")


scaler mean of feature 0: pooled (leak) 1.495 | train only (ok) 0.995
model coefficients:        leaky  [10.741 -0.114] | correct [ 9.864 -0.105]
test accuracy:             leaky  0.548 | correct 0.548


**Interpretation.** The accuracy is identical and the coefficients barely differ. Does that make the
leaky protocol acceptable? Note that the pooled scaler's mean (1.50) was computed *with the test rows
included* — the preprocessing has literally seen the test distribution. Explain why the leak is wrong in
principle, why it can be invisible in a single accuracy number, and in which situations the bias could
become large (think: imputation, feature selection, outlier clipping — anything data-dependent).


# Питання простими словами

Уявіть, ви побачили: результат контрольної однаковий, байдуже, підглядали ви чи ні. Питання запитує: **"То що, тепер підглядати можна?"** — а потім просить пояснити три речі:

1. Чому підглядати **не можна ніколи**, навіть коли результат не змінився.
2. Чому, дивлячись тільки на **одну фінальну оцінку**, ви б і не здогадались, що хтось підглядав.
3. У яких випадках підглядання **реально зашкодить** (не як тут, де пощастило).

**Your answer:**

## 1. Чи прийнятний нечесний протокол? — НІ

Те, що результат вийшов однаковим — **збіг обставин**, а не доказ безпечності методу. Правило "не використовуй test при навчанні (включно з препроцесингом)" існує не заради конкретного числа accuracy в конкретному прикладі, а як **загальний принцип**, який має працювати завжди, у будь-якій задачі. Якщо один раз "проскочило" без шкоди — це не означає, що можна так робити постійно.

## 2. Чому це неправильно "в принципі"

У реальному житті, коли модель вже працює "наживо", **майбутніх даних ще не існує** — ви фізично не можете їх побачити заздалегідь. Test set у навчальному прикладі — це імітація цього "майбутнього". Якщо ви дозволяєте scaler'у побачити test при fit — ви порушуєте саму суть симуляції: у реальності такої можливості просто немає. Тобто справа не в тому, "нашкодило це чи ні цього разу", а в тому, що **сам процес не відтворює те, що буде відбуватись насправді**.

## 3. Чому це непомітно за однією цифрою accuracy

Тому що accuracy — це дуже "грубий", підсумковий показник. Він каже лише "скільки відсотків вгадано правильно", і **не каже нічого про те, як саме модель до цього прийшла**, чи про внутрішню механіку (які значення мала кожна ознака, які ваги отримала модель). У цьому прикладі задача виявилась настільки простою (одне число визначає клас), що навіть при "спотвореному" препроцесингу модель однаково правильно розрізняла класи. Проблема реально **існує** (scaler дійсно підглянув, mean дійсно інший — 1.495 замість 0.995), просто вона не **проявилась** у фінальній цифрі. Якби ви подивились лише на accuracy — ви б ніколи не здогадались, що щось не так.

## 4. Де таке підглядання реально нашкодить

Причина, чому тут не нашкодило: StandardScaler робить просте **лінійне** перетворення (відняти число, поділити на число), а логістична регресія однаково добре працює, хоч трохи зсунь чи розтягни дані — вона просто підбере інші коефіцієнти.

Але є методи препроцесингу, які значно **чутливіші** до точних значень даних:

- **Imputation (заповнення пропусків):** якщо порожні клітинки заповнюються середнім, порахованим з train+test, а test сильно інший — заповнені числа будуть просто неправильними, і це вже спотворить реальні значення, а не тільки їхній масштаб.
- **Feature selection (відбір ознак):** якщо ви вирішуєте, які колонки залишити, дивлячись на зв'язок із **тестовими** мітками теж — ви дозволяєте тесту підказати вам, куди дивитись. Це вже не "трохи змістити число", а **змінити саму структуру того, що бачить модель**.
- **Outlier clipping (обрізання викидів):** якщо межі "нормальності" рахуються по train+test разом, а test має інший розподіл — ви ризикуєте або неправильно обрізати нормальні test-значення, або не помітити справжні викиди.

**Чому там гірше:** ці методи **нелінійні або дискретні** (щось вибирається/викидається/замінюється повністю), на відміну від простого "відняти-поділити". Тому помилка від підглядання не компенсується сама собою, як тут вийшло з логрегресією — вона накопичується і реально псує результат.

**Snippet 3c — duplicates of the same entity across the split.**

Each of 200 patients was measured 3 times (near-duplicate rows). One pipeline splits rows at random;
the other uses a *grouped* split so that a patient's rows never end up in both parts.


In [ ]:
rng = np.random.RandomState(0)
n_patients = 200
Xp = rng.normal(size=(n_patients, 2))
yp = (Xp[:, 0] + Xp[:, 1] > 0).astype(int)
X_rows = np.repeat(Xp, 3, axis=0) + rng.normal(scale=0.02, size=(n_patients * 3, 2))
y_rows = np.repeat(yp, 3)
patient_id = np.repeat(np.arange(n_patients), 3)

# Naive random split: near-duplicate rows of the same patient land on both sides
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_rows, y_rows, test_size=0.25, random_state=42)
knn1 = KNeighborsClassifier(n_neighbors=1).fit(Xr_tr, yr_tr)
print(f"Random split,  kNN(k=1): test accuracy {accuracy_score(yr_te, knn1.predict(Xr_te)):.3f}")

# Grouped split: patients never cross the boundary
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X_rows, y_rows, groups=patient_id))
knn2 = KNeighborsClassifier(n_neighbors=1).fit(X_rows[tr_idx], y_rows[tr_idx])
print(f"Grouped split, kNN(k=1): test accuracy {accuracy_score(y_rows[te_idx], knn2.predict(X_rows[te_idx])):.3f}")


Random split,  kNN(k=1): test accuracy 1.000
Grouped split, kNN(k=1): test accuracy 0.953


**Interpretation.** The random split looks great and the grouped split looks mediocre. Which number
predicts performance on *new patients*? What is the general rule for datasets with groups (patients,
schools, sessions, time series per user)?


**Your answer:** I guess the second one predicts performance on new patients. As on random split the model has a leakage as in test data there can randomly occur data on which it was trained. It has the accuracy of 1.000... General rule for datasets with groups - never mix them!

## Discussion Questions

1. Why is the majority-class (or mean) baseline the *first* thing to compute in any classification
   (regression) experiment? Give an example where a model with 94% accuracy is worthless. - **because it is on what we will check and compare with our train results on. as well, an example where a model with 94% acc is worthless is - when we have a baseline at the lvl of 0,93 for example and accuracy after training is 0,94 means it actually haven't learned sth (baseline is high cause there MUST be a class disbalance)**
2. A large train–test gap is a symptom. What are two distinct *causes* of the gap, and how would you
   distinguish them experimentally? - **That the train data was overfitted and on real data it could not work normally; another - Distribution shift (Train і test описують різні розподіли/реальності (новий прилад, нова популяція, інший час))**
3. "The test set must never influence any decision made while building the model." List three concrete - **yes, it must not as there could be leakage: the first, when accidently copied column with data that has answers, model would not train and have very high acc; second - random split of test and train data - causes leakage ; third - data with cefficients**
   actions that violate this rule (beyond what you saw in the guided exercise).
4. Preprocessing (scaling, imputation, feature selection, outlier removal): when is it legitimate to
   look at the full dataset, and when must it be fitted inside the training fold? - **You can look at the full dataset only when a transformation is strictly row-by-row and deterministic , meaning it does not need to know anything about any other row in the table. Imagine covering the entire dataset with a sheet of paper and leaving only a single row exposed: if you can still perform the operation without missing any information—like converting text to lowercase, extracting the day of the week from a timestamp, or calculating $\log(x)$—it is completely fine to do it across all data before splitting**

  On the other hand, you must fit the transformation **strictly inside the training set** whenever you need summary statistics across multiple rows. This includes scaling, filling missing values with the mean or median, and selecting top features. If you calculate an average or a maximum using the entire dataset, numbers from the future test set secretly leak into your training process, giving you a misleadingly optimistic evaluation that fails when deployed on genuinely new data.**
  
5. Your colleague tunes hyperparameters by evaluating 200 random configurations on the *same* test set
   and reports the best one. Is the reported number a fair estimate of future performance? Why?
6. In the grouped-data example, would using k = 20 instead of k = 1 remove the problem? Why or why not?


## Challenge — design a valid experiment for grouped data

**Scenario.** A hospital wants to predict readmission from repeated visits. You receive 120 patients,
1–4 visits each (420 rows in total). A student submits the following experiment:


In [ ]:
# Student's submission (run it and look at the number)
rng = np.random.RandomState(3)
n_patients = 120
severity = rng.uniform(0, 1, n_patients)
n_visits = rng.randint(1, 5, n_patients)
rows_X, rows_y, rows_pid = [], [], []
for p in range(n_patients):
    for _ in range(n_visits[p]):
        rows_X.append([severity[p] + rng.normal(0, 0.05), rng.normal(0, 1)])
        rows_y.append(int(severity[p] > 0.6))
        rows_pid.append(p)
Xv, yv, pid = np.array(rows_X), np.array(rows_y), np.array(rows_pid)
Xv_tr, Xv_te, yv_tr, yv_te = train_test_split(Xv, yv, test_size=0.3, random_state=42)
knn = KNeighborsClassifier(n_neighbors=1).fit(Xv_tr, yv_tr)
print(f"kNN(k=1) test accuracy: {accuracy_score(yv_te, knn.predict(Xv_te)):.3f}")


kNN(k=1) test accuracy: 0.944


**Your task (diagnosis + design).**
1. Detect the flaw(s) in this experiment and explain, in one or two sentences, why the reported
   accuracy is not a trustworthy estimate for new patients.
2. Write the corrected experiment: the split must respect the `pid` grouping, and the evaluation must
   report the number that actually predicts performance on unseen patients. Use the code cell below.
3. **Counterfactual:** predict what happens to the test accuracy if you rerun the corrected experiment
   with k = 20 instead of k = 1, and say whether the change makes the estimate more or less honest.


In [ ]:
# Your code here — corrected experiment for the readmission data


**Your answer (written diagnosis and design):**

## Takeaways

- The test set is a simulation of the future: anything learned from it (directly or indirectly)
  invalidates your estimate.
- Always put a trivial baseline (majority class / mean) next to your model's number — accuracy without
  a baseline is uninterpretable.
- The train–test gap is the standard overfitting diagnostic: report both numbers, always.
- Preprocessing (scaling, imputation, feature selection) must be fitted on the training part only and
  applied to the test part — never on the pooled data.
- Datasets with groups (patients, schools, users, sessions) need grouped splits such as
  `GroupShuffleSplit` / `GroupKFold`; random row splits leak near-duplicates across the boundary.
- Suspiciously good numbers are the first symptom of leakage — audit features for target-derived
  information before trusting an accuracy value.
